# JPmart - Silver Transformation

Reads Bronze tables (landed as-is, all `string` columns) and produces cleaned, typed Silver tables. Each entity is split into two outputs:

- **`<entity>`** — valid, cleaned rows.
- **`<entity>_quarantine`** — rows with a critical, unrecoverable problem (missing primary key, or a missing/invalid required foreign key), kept for auditing instead of silently dropped.

**Rule of thumb:** a *recoverable formatting issue* (bad casing, mixed date formats, a stray currency symbol) is cleaned in place and the row stays. Only a problem that invalidates the row's identity or its ability to be
linked to other data sends it to quarantine.

## Configuration

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, upper, coalesce, try_to_date, lit, when, regexp_replace
)

CATALOG = "jpmart"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Every format dirty_date() can produce in 00_generate_fake_data, in Spark date-pattern syntax. Used with coalesce() so parsing tries each format in
# order and keeps the first one that succeeds.
DATE_FORMATS = ["yyyy-MM-dd", "dd/MM/yyyy", "MM-dd-yyyy", "yyyy/MM/dd", "MMMM d, yyyy"]

## Shared helper — write Silver + Quarantine

In [0]:
def write_silver_tables(valid_df, quarantine_df, entity_name):
    """Writes cleaned rows to jpmart.silver.<entity_name>, and rows that fail a critical validation (missing/invalid key) to
    jpmart.silver.<entity_name>_quarantine for later auditing."""
    target_table = f"{CATALOG}.{SILVER_SCHEMA}.{entity_name}"
    quarantine_table = f"{CATALOG}.{SILVER_SCHEMA}.{entity_name}_quarantine"

    (
        valid_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    print(f"  -> {target_table}: {valid_df.count():,} rows")

    (
        quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(quarantine_table)
    )
    print(f"  -> {quarantine_table}: {quarantine_df.count():,} rows quarantined")

## Cleaning functions

One function per entity, each returning `(silver_df, quarantine_df)`.

In [0]:
def clean_customers(bronze_df):
    """Cleans Bronze customers into Silver + quarantine.

    Quarantine rule: customer_id is the primary key. The generator never actually leaves it null, but the check stays in as a defensive guard
    rather than an assumption that upstream will always behave -- the same principle applied to every entity below.

    Everything else here is a recoverable formatting issue, not a structural problem, so it's cleaned in place and the row stays:
    - email: unrecoverable formats (missing "@") become null rather than guessed at; recoverable ones (extra whitespace/casing) are normalized.
    - state: normalized to uppercase.
    - signup_date: parsed against every format the generator can produce.
    - loyalty_member: cast to boolean.
    """
    is_valid = col("customer_id").isNotNull()

    quarantine = (
        bronze_df
        .filter(~is_valid)
        .withColumn("_quarantine_reason", lit("missing customer_id"))
    )

    silver = (
        bronze_df
        .filter(is_valid)
        .withColumn(
            "email",
            when(col("email").contains("@"), lower(trim(col("email"))))
            .otherwise(lit(None))
        )
        .withColumn("state", upper(trim(col("state"))))
        .withColumn(
            "signup_date",
            coalesce(*[try_to_date(col("signup_date"), fmt) for fmt in DATE_FORMATS])
        )
        .withColumn("loyalty_member", lower(trim(col("loyalty_member"))).cast("boolean"))
    )

    return silver, quarantine

def clean_products(bronze_df):
    """Cleans Bronze products into Silver + quarantine.

    Quarantine rule: product_id is the primary key -- same defensive check as every other entity.

    unit_price arrives dirty three ways: "$123.45" (string), negative (capture error), or null. The currency symbol is recoverable and
    stripped; a negative value is NOT recoverable (there's no way to know what the intended price was), so it's nulled out rather than guessed
    at with abs(). A product with an unknown price is still a valid product -- it stays in Silver, not quarantine.

    category null is left as-is: it's missing information, not a formatting error, and there's no "correct" value to reconstruct.

    active is normalized from its bool/"yes"/"no" inconsistency into a real boolean; anything unexpected becomes null rather than a guess.
    """
    is_valid = col("product_id").isNotNull()

    quarantine = (
        bronze_df
        .filter(~is_valid)
        .withColumn("_quarantine_reason", lit("missing product_id"))
    )

    clean_price = regexp_replace(col("unit_price"), "[$]", "").cast("double")

    silver = (
        bronze_df
        .filter(is_valid)
        .withColumn("unit_price", when(clean_price < 0, lit(None)).otherwise(clean_price))
        .withColumn(
            "active",
            when(lower(trim(col("active"))).isin("true", "yes"), lit(True))
            .when(lower(trim(col("active"))).isin("false", "no"), lit(False))
            .otherwise(lit(None))
        )
    )

    return silver, quarantine

def clean_orders(bronze_df):
    """Cleans Bronze orders into Silver + quarantine.

    Quarantine rule: customer_id is a required FK -- an order that can't be attributed to a customer isn't usable downstream, so it's
    quarantined rather than kept with a null.

    Note on duplicates: unlike order_items below, the ~1.5% exact-duplicate rows the generator injects never actually reach this table as
    duplicates. upsert_batch() in 01_ingest_bronze deduplicates every micro-batch by order_id before writing/merging (MERGE INTO rejects
    multiple source rows per key). The dropDuplicates() call below is a defensive safety net, not doing real work today.
    """
    is_valid = col("customer_id").isNotNull() & col("order_id").isNotNull()

    quarantine = (
        bronze_df
        .filter(~is_valid)
        .withColumn(
            "_quarantine_reason",
            when(col("order_id").isNull(), lit("missing order_id"))
            .otherwise(lit("missing customer_id"))
        )
    )

    silver = (
        bronze_df
        .filter(is_valid)
        .dropDuplicates(["order_id"])  # defensive; see note above
        .withColumn("status", lower(trim(col("status"))))
        .withColumn(
            "order_date",
            coalesce(*[try_to_date(col("order_date"), fmt) for fmt in DATE_FORMATS])
        )
    )

    return silver, quarantine

def clean_order_items(bronze_df, valid_orders_df):
    """Cleans Bronze order_items into Silver + quarantine.

    Two quarantine reasons now, not one:
    - missing a required field (order_item_id/order_id/product_id)
    - order_id doesn't match any order that survived Silver cleaning
      (i.e. its parent order was itself quarantined, e.g. for a missing
      customer_id) -- an item on an unresolvable order isn't usable
      downstream either, so it cascades into quarantine too.

    Checked via a join against valid_orders_df rather than col.isin() with
    a collected Python list -- isin() with thousands of literal values
    doesn't scale and bloats the query plan; a join does the same check
    as a distributed operation.
    """
    has_required_fields = (
        col("order_item_id").isNotNull()
        & col("order_id").isNotNull()
        & col("product_id").isNotNull()
    )

    missing_fields = (
        bronze_df.filter(~has_required_fields)
        .withColumn("_quarantine_reason", lit("missing product_id"))
    )

    candidates = bronze_df.filter(has_required_fields)
    valid_order_ids = valid_orders_df.select("order_id")

    matched = candidates.join(valid_order_ids, on="order_id", how="left_semi")
    orphaned_order = (
        candidates.join(valid_order_ids, on="order_id", how="left_anti")
        .withColumn("_quarantine_reason", lit("order_id not found in valid orders"))
    )

    quarantine = missing_fields.unionByName(orphaned_order)

    clean_quantity = col("quantity").cast("int")
    clean_price = regexp_replace(col("unit_price"), "[$]", "").cast("double")

    silver = (
        matched
        .dropDuplicates(["order_item_id"])
        .withColumn("quantity", when(clean_quantity < 0, lit(None)).otherwise(clean_quantity))
        .withColumn("unit_price", when(clean_price < 0, lit(None)).otherwise(clean_price))
    )

    return silver, quarantine

def clean_web_events(bronze_df):
    """Cleans Bronze web_events into Silver + quarantine.

    Quarantine rule: event_id is the only structural requirement. It's never actually dirtied by the generator, but the check stays in as a
    defensive guard, same as every other entity.

    Null customer_id is NOT a quality problem here -- it represents a genuine anonymous session, a normal state in real clickstream data.
    Null product_id is left as-is too: many event types (e.g. a page_view on the homepage) legitimately have no associated product. Neither
    gets cleaned or quarantined.

    event_timestamp is parsed with to_date, not to_timestamp: the generator's dirty_date() helper always formats with date-only
    patterns, so no time-of-day component actually survives -- the true granularity of this column is daily.
    """
    is_valid = col("event_id").isNotNull()

    quarantine = (
        bronze_df
        .filter(~is_valid)
        .withColumn("_quarantine_reason", lit("missing event_id"))
    )

    silver = (
        bronze_df
        .filter(is_valid)
        .withColumn(
            "event_timestamp",
            coalesce(*[try_to_date(col("event_timestamp"), fmt) for fmt in DATE_FORMATS])
        )
    )

    return silver, quarantine

## Run all cleaning functions

In [0]:
# orders must run before order_items -- order_items' validation now
# depends on which orders survived Silver cleaning (see clean_order_items)
bronze_orders = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders")
orders_silver, orders_quarantine = clean_orders(bronze_orders)
write_silver_tables(orders_silver, orders_quarantine, "orders")

CLEANING_FUNCTIONS = [
    ("customers", clean_customers),
    ("products", clean_products),
    ("web_events", clean_web_events),
]

for entity_name, clean_fn in CLEANING_FUNCTIONS:
    bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{entity_name}")
    silver_df, quarantine_df = clean_fn(bronze_df)
    write_silver_tables(silver_df, quarantine_df, entity_name)

bronze_order_items = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_items")
order_items_silver, order_items_quarantine = clean_order_items(bronze_order_items, orders_silver)
write_silver_tables(order_items_silver, order_items_quarantine, "order_items")

## Validation

Confirm every Bronze row was accounted for (Silver + quarantine), and
spot-check that the specific dirty patterns were actually resolved.

In [0]:
for entity_name, _ in CLEANING_FUNCTIONS:
    bronze_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{entity_name}").count()
    silver_count = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{entity_name}").count()
    quarantine_count = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{entity_name}_quarantine").count()
    gap = bronze_count - (silver_count + quarantine_count)

    # order_items is append-only and never deduplicated upstream (unlike customers/orders, which are deduped by the Bronze MERGE), so its gap
    # is expected: it's exact-duplicate rows removed by dropDuplicates(), not data loss. Any other entity should always show gap == 0.
    label = "duplicates removed" if entity_name == "order_items" else "unaccounted"

    print(
        f"{entity_name}: {bronze_count:,} bronze -> "
        f"{silver_count:,} silver + {quarantine_count:,} quarantined "
        f"({gap:,} {label})"
    )

    if entity_name != "order_items":
        assert gap == 0, f"{entity_name}: unexpected row loss ({gap} rows)"